In [1]:
import tempfile
import os
from pathlib import Path
import re

# 📍【关键修复 1】：重定向 Catalyst 的临时文件，彻底解决超算节点硬盘爆满的问题！
custom_tmp_path = os.path.join(os.getcwd(), "catalyst_tmp_cache")
os.makedirs(custom_tmp_path, exist_ok=True)
os.environ['TMPDIR'] = custom_tmp_path
tempfile.tempdir = custom_tmp_path
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import time
import math
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
from scipy.sparse import coo_matrix

import jax
# 开启双精度
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import pennylane as qml
import catalyst
qml.about()


Name: pennylane
Version: 0.43.1
Summary: PennyLane is a cross-platform Python library for quantum computing, quantum machine learning, and quantum chemistry. Train a quantum computer the same way as a neural network.
Home-page: 
Author: 
Author-email: 
License-Expression: Apache-2.0
Location: /home/lzs/.conda/envs/lzsgpu/lib/python3.12/site-packages
Requires: appdirs, autograd, autoray, cachetools, diastatic-malt, networkx, numpy, packaging, pennylane-lightning, requests, rustworkx, scipy, tomlkit, typing_extensions
Required-by: pennylane_catalyst, pennylane_lightning, pennylane_lightning_gpu

Platform info:           Linux-6.11.0-26-generic-x86_64-with-glibc2.39
Python version:          3.12.12
Numpy version:           2.3.4
Scipy version:           1.16.2
JAX version:             0.6.2
Installed devices:
- default.clifford (pennylane-0.43.1)
- default.gaussian (pennylane-0.43.1)
- default.mixed (pennylane-0.43.1)
- default.qubit (pennylane-0.43.1)
- default.qutrit (pennylane-0.43.1)


In [2]:
# =================== 0. 环境与硬件检查 ===================
print('✅ JAX version:', jax.__version__)
print('✅ Devices:', jax.devices())
if any(d.platform == 'gpu' for d in jax.devices()):
    print('🎉 GPU is working!')
else:
    print('⚠️ No GPU detected')


✅ JAX version: 0.6.2
✅ Devices: [CudaDevice(id=0)]
🎉 GPU is working!


In [3]:
# =================== 1. 数据准备 (直接使用 A100 的大显存优势) ===================
target_l, target_n = 5, 4
matrix_name_candidates = [
    f"L={target_l} N={target_n}.npz",
    f"L={target_l} n={target_n}.npz",
    f"l={target_l} N={target_n}.npz",
    f"l={target_l} n={target_n}.npz",
]
matrix_name = matrix_name_candidates[0]
cwd = Path.cwd()

def _normalize_filename(name: str) -> str:
    return "".join(name.lower().split())

matrix_file = None
matrix_override = os.environ.get("MATRIX_FILE")
if matrix_override:
    override_path = Path(matrix_override).expanduser()
    if override_path.exists():
        matrix_file = override_path
    else:
        print(f"⚠️ MATRIX_FILE 不存在: {override_path}")

candidate_roots = [cwd, cwd / "lunwen", cwd.parent, cwd.parent / "lunwen"]
matrix_candidates = [root / name for root in candidate_roots for name in matrix_name_candidates]
if matrix_file is None:
    matrix_file = next((p for p in matrix_candidates if p.exists()), None)

search_roots = [cwd, cwd.parent, cwd.parent.parent]
if matrix_file is None:
    for root in search_roots:
        if not root.exists():
            continue
        for name in matrix_name_candidates:
            matches = sorted(root.rglob(name))
            if matches:
                matrix_file = matches[0]
                break
        if matrix_file is not None:
            break

if matrix_file is None:
    expected_norms = {_normalize_filename(name) for name in matrix_name_candidates}
    fuzzy_hits = []
    npz_inventory = []
    for root in search_roots:
        if not root.exists():
            continue
        for p in root.rglob("*.npz"):
            npz_inventory.append(p)
            name_norm = _normalize_filename(p.name)
            if name_norm in expected_norms:
                fuzzy_hits.append(p)
                continue
            has_l = re.search(rf"l\\s*=\\s*{target_l}\\b", p.name, flags=re.IGNORECASE)
            has_n = re.search(rf"n\\s*=\\s*{target_n}\\b", p.name, flags=re.IGNORECASE)
            if has_l and has_n:
                fuzzy_hits.append(p)

    if fuzzy_hits:
        matrix_file = sorted(set(fuzzy_hits), key=lambda x: (len(str(x)), str(x)))[0]
    else:
        tried = [str(p) for p in matrix_candidates]
        sample = [str(p) for p in sorted(set(npz_inventory), key=lambda x: str(x))[:20]]
        scanned = [str(p) for p in search_roots]
        raise FileNotFoundError(
            f"Cannot find matrix file for L=5,N=4. cwd={cwd}, tried={tried}, scanned={scanned}, npz_sample={sample}"
        )
H_3 = sp.load_npz(str(matrix_file))
print(f"使用矩阵文件: {matrix_file}")

min_eigvals, _ = eigsh(H_3, k=1, which='SA')
exact_min_energy = float(min_eigvals[0])
print(f"最小特征值 (理论极限): {exact_min_energy:.8f}")

d = H_3.shape[0]
n_qubits = int(np.ceil(np.log2(d)))
l = 2 ** n_qubits
depth = math.ceil(2**n_qubits / n_qubits) + n_qubits
print(f'电路深度: {depth}, 参数总量: {depth * n_qubits}')

# 补全到 2^Nq 并 Gray 编码 (逻辑和之前一样)
H_3_coo = H_3.tocoo()
rows, cols, data = H_3_coo.row.astype(np.int64), H_3_coo.col.astype(np.int64), H_3_coo.data
if l > d:
    rows = np.concatenate([rows, np.arange(d, l, dtype=np.int64)])
    cols = np.concatenate([cols, np.arange(d, l, dtype=np.int64)])
    data = np.concatenate([data, np.full(l - d, 1000.0, dtype=data.dtype)])

def gray_code(n):
    if n == 1: return ["0", "1"]
    lower = gray_code(n - 1)
    return ["0" + x for x in lower] + ["1" + x for x in reversed(lower)]

gray_basis = gray_code(n_qubits)
gray2natural = np.array([int(g, 2) for g in gray_basis], dtype=np.int64)
new_rows, new_cols = gray2natural[rows], gray2natural[cols]
H_gray_csr = coo_matrix((data, (new_rows, new_cols)), shape=(l, l)).tocsr()

# 📍【关键修复 2】：A100 显存管够！直接将 4GB 的矩阵转化为 JAX 密集数组传入 GPU，速度拉满！
H_dense = H_gray_csr.toarray()
H_jax = jnp.array(H_dense, dtype=jnp.complex128)
del H_3, H_3_coo, H_gray_csr, H_dense


使用矩阵文件: /mnt/data/lzs/project/lunwen/L=5 n=4.npz
最小特征值 (理论极限): -35.68407846
电路深度: 354, 参数总量: 4248


In [4]:
# =================== 2. 量子电路与 Catalyst 编译（L-BFGS-B） ===================
# 构造参数化电路并提供 L-BFGS-B 所需的能量与梯度函数。
hf = np.zeros(n_qubits, dtype=int)
wires = list(range(n_qubits))
dev = qml.device("lightning.gpu", wires=n_qubits)

@qml.qnode(dev)
def cost_circuit(params_2d, h_mat):
    qml.BasisState(hf, wires=wires)

    for w in wires:
        qml.Hadamard(wires=w)

    for layer in range(depth):
        for w in wires:
            qml.RY(params_2d[layer, w], wires=w)

        # 环形+交错纠缠
        for w in range(0, n_qubits - 1, 2):
            qml.CNOT(wires=[w, w + 1])
        qml.CNOT(wires=[n_qubits - 1, 0])
        for w in range(1, n_qubits - 1, 2):
            qml.CNOT(wires=[w, w + 1])

    return qml.expval(qml.Hermitian(h_mat, wires=wires))

# 小随机初始化，避免全零点附近的对称平坦区域
key = jax.random.PRNGKey(42)
init_params = 0.005 * jax.random.normal(key, shape=(depth, n_qubits), dtype=jnp.float64)

@qml.qjit(autograph=True)
def lbfgs_value_grad(flat_params, h_matrix):
    params_2d = jnp.reshape(flat_params, (depth, n_qubits))

    def cost_fn(p):
        return cost_circuit(p, h_matrix)

    energy, grads = catalyst.value_and_grad(cost_fn)(params_2d)
    return energy, jnp.reshape(grads, (depth * n_qubits,))


In [ ]:
# =================== 3. 纯 L-BFGS-B 训练（精简版） ===================
from scipy.optimize import minimize

target_gap = 1e-5
x0 = np.asarray(init_params, dtype=np.float64).reshape(-1)

# 可选边界：旋转角限制在 [-pi, pi]，可抑制无效大步长搜索
use_param_bounds = True
param_bound = np.pi
bounds = [(-param_bound, param_bound)] * x0.size if use_param_bounds else None

lbfgs_probe_enable = True
lbfgs_probe_maxiter = 12
lbfgs_maxiter = 8000
lbfgs_maxcor = 100
lbfgs_maxls = 80
lbfgs_ftol = 1e-15
lbfgs_gtol = 1e-12
lbfgs_log_every = 20

lbfgs_energy_history = []
lbfgs_stop_on_target_gap = True
lbfgs_stop_gap = target_gap
lbfgs_stop_state = {"triggered": False, "x": None, "f": None}

print("")
print("🚀 开始纯 L-BFGS-B 训练（精简版）...")
print(
    f"L-BFGS-B 配置: maxiter={lbfgs_maxiter}, ftol={lbfgs_ftol:.1e}, "
    f"gtol={lbfgs_gtol:.1e}, maxcor={lbfgs_maxcor}, maxls={lbfgs_maxls}, target_gap={target_gap:.1e}"
)

start_time = time.time()

# 缓存 fun/jac，减少 scipy 在同一点重复调用带来的开销
cache = {"x": None, "f": None, "g": None}

def _fun_jac_cached(x_np):
    x_cached = cache["x"]
    if x_cached is None or x_cached.shape != x_np.shape or not np.array_equal(x_cached, x_np):
        x_jax = jnp.asarray(x_np, dtype=jnp.float64)
        e_val, g_val = lbfgs_value_grad(x_jax, H_jax)
        cache["x"] = np.array(x_np, copy=True)
        cache["f"] = float(e_val)
        cache["g"] = np.asarray(g_val, dtype=np.float64)
    return cache["f"], cache["g"]

def _scipy_fun(x_np):
    f_val, _ = _fun_jac_cached(x_np)
    return f_val

def _scipy_jac(x_np):
    _, g_val = _fun_jac_cached(x_np)
    return g_val

def _scipy_callback(xk):
    f_val, _ = _fun_jac_cached(xk)
    lbfgs_energy_history.append(float(f_val))
    gap_now = abs(f_val - exact_min_energy)

    if len(lbfgs_energy_history) == 1 or len(lbfgs_energy_history) % lbfgs_log_every == 0:
        print(
            f"[L-BFGS] iter={len(lbfgs_energy_history):4d}, "
            f"Energy={f_val:.10f} Ha, Gap={gap_now:.6e}"
        )

    if lbfgs_stop_on_target_gap and gap_now <= lbfgs_stop_gap:
        lbfgs_stop_state["triggered"] = True
        lbfgs_stop_state["x"] = np.array(xk, dtype=np.float64, copy=True)
        lbfgs_stop_state["f"] = float(f_val)
        print(f"[L-BFGS] 提前停止: gap={gap_now:.6e} <= target_gap={lbfgs_stop_gap:.1e}")
        raise StopIteration

f0, _ = _fun_jac_cached(x0)
f0_gap = abs(f0 - exact_min_energy)
print(f"[L-BFGS] 初始能量: {f0:.10f} Ha, Gap={f0_gap:.6e}")

energy_history = [float(f0)]
best_energy = float(f0)
best_gap = float(f0_gap)
best_step = 0
best_params = jnp.asarray(x0, dtype=jnp.float64).reshape((depth, n_qubits))

lbfgs_enable = f0_gap > target_gap

# 先做短探针，确认数值稳定后再跑长程优化
if lbfgs_enable and lbfgs_probe_enable:
    probe_res = minimize(
        fun=_scipy_fun,
        x0=x0,
        jac=_scipy_jac,
        method='L-BFGS-B',
        bounds=bounds,
        options={
            'maxiter': lbfgs_probe_maxiter,
            'maxcor': min(20, lbfgs_maxcor),
            'maxls': min(40, lbfgs_maxls),
            'ftol': 1e-9,
            'gtol': 1e-8,
        },
    )

    probe_f = float(probe_res.fun)
    probe_gap = abs(probe_f - exact_min_energy)
    probe_delta = f0 - probe_f
    print(
        f"[L-BFGS Probe] success={probe_res.success}, nit={probe_res.nit}, "
        f"Energy={probe_f:.10f} Ha, Gap={probe_gap:.6e}, ΔE={probe_delta:.3e}"
    )

    probe_ok = np.isfinite(probe_f) and (probe_delta > -1e-8)
    if probe_ok:
        x0 = np.asarray(probe_res.x, dtype=np.float64)
    else:
        lbfgs_enable = False
        print("⚠️ L-BFGS 探针未通过，跳过长程 L-BFGS-B 精修。")

if lbfgs_enable:
    lbfgs_stopped_early = False
    try:
        res = minimize(
            fun=_scipy_fun,
            x0=x0,
            jac=_scipy_jac,
            method='L-BFGS-B',
            bounds=bounds,
            callback=_scipy_callback,
            options={
                'maxiter': lbfgs_maxiter,
                'maxcor': lbfgs_maxcor,
                'maxls': lbfgs_maxls,
                'ftol': lbfgs_ftol,
                'gtol': lbfgs_gtol,
            },
        )
    except StopIteration:
        lbfgs_stopped_early = lbfgs_stop_state["triggered"]
        res = None

    if lbfgs_stopped_early:
        lbfgs_final_energy = float(lbfgs_stop_state["f"])
        lbfgs_x = np.asarray(lbfgs_stop_state["x"], dtype=np.float64)
        lbfgs_success = True
        lbfgs_status = 99
        lbfgs_nit = len(lbfgs_energy_history)
        lbfgs_nfev = lbfgs_nit
        lbfgs_njev = lbfgs_nit
    else:
        lbfgs_final_energy = float(res.fun)
        lbfgs_x = np.asarray(res.x, dtype=np.float64)
        lbfgs_success = bool(res.success)
        lbfgs_status = int(res.status)
        lbfgs_nit = int(res.nit)
        lbfgs_nfev = int(res.nfev)
        lbfgs_njev = int(res.njev)

    lbfgs_final_gap = abs(lbfgs_final_energy - exact_min_energy)

    if (not lbfgs_energy_history) or abs(lbfgs_energy_history[-1] - lbfgs_final_energy) > 1e-15:
        lbfgs_energy_history.append(lbfgs_final_energy)
    energy_history.extend(lbfgs_energy_history)

    if lbfgs_final_energy < best_energy:
        best_energy = lbfgs_final_energy
        best_gap = lbfgs_final_gap
        best_step = len(energy_history) - 1
        best_params = jnp.asarray(lbfgs_x, dtype=jnp.float64).reshape((depth, n_qubits))

    params = best_params
    current_energy = lbfgs_final_energy

    print(
        f"[L-BFGS] 结束: success={lbfgs_success}, status={lbfgs_status}, "
        f"nit={lbfgs_nit}, nfev={lbfgs_nfev}, njev={lbfgs_njev}"
    )
    print(f"[L-BFGS] 最终能量: {lbfgs_final_energy:.10f} Ha, Gap={lbfgs_final_gap:.6e}")
else:
    current_energy = float(f0)
    print("\nℹ️ 初始参数已达到目标精度，跳过长程 L-BFGS-B 精修。")

energy_array = np.array(energy_history, dtype=np.float64)

print("-" * 30)
print(f"训练总耗时: {time.time() - start_time:.2f} 秒")
print(f"最后一步能量: {current_energy:.10f} Ha")
print(f"历史最优能量: {best_energy:.10f} Ha (step={best_step})")
print(f"最优能量差(best_gap): {best_gap:.8e} Ha")
print(f"目标精度(target_gap): {target_gap:.1e} Ha")
print(f"energy_array 已生成，shape={energy_array.shape}")



🚀 开始纯 L-BFGS-B 训练（精简版）...
L-BFGS-B 配置: maxiter=8000, ftol=1.0e-15, gtol=1.0e-12, maxcor=100, maxls=80, target_gap=1.0e-05
[L-BFGS] 初始能量: 480.0974332613 Ha, Gap=5.157815e+02


In [ ]:
# =================== 4. 能量差距对数图（Log） ===================
import numpy as np
import matplotlib.pyplot as plt

if 'energy_array' not in globals():
    raise RuntimeError("未检测到 energy_array，请先运行训练单元。")

energy_plot = np.asarray(energy_array, dtype=np.float64)
if energy_plot.size == 0:
    raise RuntimeError("energy_array 为空，无法绘图。")

# 使用 1..N 作为横轴，便于在需要时切换 log-x
steps_axis = np.arange(1, energy_plot.size + 1)

# 历史最优包络线（单调不增）
best_so_far = np.minimum.accumulate(energy_plot)

# 构造可取对数的 gap（必须 > 0）
eps = 1e-14
if 'exact_min_energy' in globals():
    ref_energy = float(exact_min_energy)
    ref_name = 'exact_min_energy'
    gap = np.maximum(energy_plot - ref_energy, eps)
    best_gap = np.maximum(best_so_far - ref_energy, eps)
else:
    # 若没有精确值，则以最终最优值作参考
    ref_energy = float(best_so_far[-1])
    ref_name = 'best_final'
    gap = np.maximum(energy_plot - ref_energy + eps, eps)
    best_gap = np.maximum(best_so_far - ref_energy + eps, eps)

# 后段放大，观察精修阶段
trail_n = max(200, energy_plot.size // 4)
start_tail = max(0, energy_plot.size - trail_n)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=120)

# 左图：全程 gap 对数图（y-log）
axes[0].semilogy(steps_axis, gap, lw=1.2, alpha=0.8, label='Gap (Energy - Ref)')
axes[0].semilogy(steps_axis, best_gap, lw=1.4, label='Best Gap')
axes[0].set_title(f'Log Gap vs Step (Full), ref={ref_name}')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Gap (Ha, log scale)')
axes[0].grid(alpha=0.25, which='both')
axes[0].legend()

# 右图：后段 gap 对数图（y-log）
axes[1].semilogy(steps_axis[start_tail:], gap[start_tail:], lw=1.2, alpha=0.85, label='Gap (tail)')
axes[1].semilogy(steps_axis[start_tail:], best_gap[start_tail:], lw=1.4, label='Best Gap (tail)')
axes[1].set_title(f'Log Gap vs Step (Last {energy_plot.size - start_tail} Steps)')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Gap (Ha, log scale)')
axes[1].grid(alpha=0.25, which='both')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"参考能量(ref): {ref_energy:.10f} Ha ({ref_name})")
print(f"最终能量: {energy_plot[-1]:.10f} Ha")
print(f"历史最优: {best_so_far[-1]:.10f} Ha")
print(f"最终 gap: {gap[-1]:.10e} Ha")
print(f"最优 gap: {best_gap[-1]:.10e} Ha")
